# Scraping PriceCharting for the Pokémon card dataset

This notebook pulls Pokémon card prices from [pricecharting.com](https://www.pricecharting.com).

It does two things:

1. **Current prices** — for every set (e.g. `pokemon-promo`), walk the listing table
   and record each card's price at every grade (Ungraded, Grade 7/8/9/9.5, PSA 10).
2. **Price history** — for a card's product page, pull the monthly price-over-time
   series per grade (this is what powers the scrollytelling chart).

**Please scrape responsibly:** there's a polite delay between requests, retries with
backoff, and a normal browser User-Agent. The full dataset is ~300 sets and tens of
thousands of cards — start with a single set, confirm the output, then scale up.
Check PriceCharting's Terms of Service before running the whole thing.


### Setup

In [1]:
# !pip install requests beautifulsoup4 pandas lxml
import time, re, json, random
from datetime import datetime, timezone

import requests
import pandas as pd
from bs4 import BeautifulSoup

BASE = "https://www.pricecharting.com"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"
}
DELAY = 1.5   # base seconds to wait between requests — keep this polite

session = requests.Session()
session.headers.update(HEADERS)


def fetch(url, method="GET", data=None, retries=3):
    """GET/POST with retry + backoff, then sleep so we don't hammer the server."""
    for attempt in range(retries):
        try:
            r = session.request(method, url, data=data, timeout=30)
            r.raise_for_status()
            time.sleep(DELAY + random.uniform(0, 0.6))
            return r.text
        except requests.RequestException as e:
            wait = 2 ** attempt
            print(f"  ! {e} -> retry in {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"failed after {retries} tries: {url}")


def money(text):
    """'$2,800.00' -> 2800.0 ; blanks/dashes -> None."""
    if not text:
        return None
    t = text.replace("$", "").replace(",", "").strip()
    if t in ("", "-", "N/A"):
        return None
    try:
        return float(t)
    except ValueError:
        return None


### 1. Discover every Pokémon set\n\nThe category page links out to each set's `/console/pokemon-*` listing.

In [2]:
def discover_sets():
    html = fetch(f"{BASE}/category/pokemon-cards")
    slugs = sorted(set(re.findall(r"/console/(pokemon[a-z0-9-]+)", html)))
    print(f"found {len(slugs)} pokemon sets")
    return slugs

all_sets = discover_sets()
all_sets[:10]


found 299 pokemon sets


['pokemon-1999-topps-movie',
 'pokemon-1999-topps-tv',
 'pokemon-2000-topps-chrome',
 'pokemon-2000-topps-tv',
 'pokemon-2000-topps-tv-episode',
 'pokemon-2001-topps-johto',
 'pokemon-2001-topps-johto-sticker',
 'pokemon-2003-topps-advanced',
 'pokemon-2003-topps-advanced-die-cut',
 'pokemon-2004-starter-deck']

### 2. Scrape one set's current prices

Each listing lives in `table#games_table`. The visible grade columns vary by set,
so we read the `<thead>` labels and zip them onto each row's price cells. The
"more results" button is a POST form carrying a `cursor` — we follow it until the
last page.

In [3]:
def scrape_set(slug):
    url = f"{BASE}/console/{slug}"
    rows, labels, seen_cursors = [], None, set()
    data = None  # first request is a plain GET; later pages POST the cursor form

    while True:
        html = fetch(url, method="POST", data=data) if data else fetch(url)
        soup = BeautifulSoup(html, "lxml")
        table = soup.find("table", id="games_table")
        if table is None:
            break

        # grade column labels (drop the blank image column and the 'Card' column)
        if labels is None:
            ths = [th.get_text(strip=True) for th in table.select("thead th")]
            labels = [t for t in ths if t and t.lower() != "card"]

        new_rows = 0
        for tr in table.select('tbody tr[id^="product-"]'):
            a = tr.select_one("td.title a")
            if not a:
                continue
            href = a["href"]
            prices = [money(s.get_text(strip=True))
                      for s in tr.select("td.price span.js-price")]
            rec = {
                "set": slug,
                "product_id": tr.get("data-product"),
                "card": a.get_text(strip=True),
                "url": BASE + href if href.startswith("/") else href,
            }
            for label, price in zip(labels, prices):
                rec[label] = price
            rows.append(rec)
            new_rows += 1

        # follow pagination
        form = soup.select_one("form.js-next-page")
        cursor_inp = form.find("input", {"name": "cursor"}) if form else None
        if not cursor_inp or new_rows == 0:
            break
        cursor = cursor_inp.get("value")
        if cursor in seen_cursors:
            break
        seen_cursors.add(cursor)
        data = {inp.get("name"): inp.get("value", "")
                for inp in form.select("input")
                if inp.get("name") and inp.get("type") != "submit"}

    print(f"  {slug}: {len(rows)} cards")
    return rows

# quick test on the promo set the story uses
promo = scrape_set("pokemon-promo")
pd.DataFrame(promo).head()


  pokemon-promo: 2482 cards


,set,product_id,card,url,Ungraded,Grade 9,PSA 10
0,pokemon-promo,5834844,Pikachu with Grey Felt Hat #85,https://www.pricecharting.com/game/pokemon-pro...,922.00,974.00,2800.0
1,pokemon-promo,11197498,Mega Charizard X EX #23,https://www.pricecharting.com/game/pokemon-pro...,40.12,52.15,221.1
2,pokemon-promo,1958438,Ancient Mew,https://www.pricecharting.com/game/pokemon-pro...,112.50,251.69,2979.0
3,pokemon-promo,11239139,N's Zekrom #31,https://www.pricecharting.com/game/pokemon-pro...,12.30,43.51,476.5
4,pokemon-promo,4246447,Charizard VStar #SWSH262,https://www.pricecharting.com/game/pokemon-pro...,80.00,137.43,1675.0


### 3. Scrape all sets \u2192 `pokemon_prices.csv`\n\nStart small (`all_sets[:3]`) to sanity-check, then run the full list.

In [4]:
SETS_TO_SCRAPE = all_sets          # or a slice like all_sets[:3] to test
# SETS_TO_SCRAPE = ["pokemon-promo"]

all_rows = []
for i, slug in enumerate(SETS_TO_SCRAPE, 1):
    print(f"[{i}/{len(SETS_TO_SCRAPE)}] {slug}")
    try:
        all_rows += scrape_set(slug)
    except Exception as e:
        print(f"  !! skipped {slug}: {e}")

prices = pd.DataFrame(all_rows)
import os; os.makedirs("data", exist_ok=True)
prices.to_csv("data/pokemon_prices.csv", index=False)
print("saved data/pokemon_prices.csv", prices.shape)
prices.head()


[1/299] pokemon-1999-topps-movie
  pokemon-1999-topps-movie: 201 cards
[2/299] pokemon-1999-topps-tv
  pokemon-1999-topps-tv: 290 cards
[3/299] pokemon-2000-topps-chrome
  pokemon-2000-topps-chrome: 607 cards
[4/299] pokemon-2000-topps-tv
  pokemon-2000-topps-tv: 320 cards
[5/299] pokemon-2000-topps-tv-episode
  pokemon-2000-topps-tv-episode: 57 cards
[6/299] pokemon-2001-topps-johto
  pokemon-2001-topps-johto: 130 cards
[7/299] pokemon-2001-topps-johto-sticker
  pokemon-2001-topps-johto-sticker: 62 cards
[8/299] pokemon-2003-topps-advanced
  pokemon-2003-topps-advanced: 180 cards
[9/299] pokemon-2003-topps-advanced-die-cut
  pokemon-2003-topps-advanced-die-cut: 18 cards
[10/299] pokemon-2004-starter-deck
  pokemon-2004-starter-deck: 41 cards
[11/299] pokemon-2005-quick-construction-pack
  pokemon-2005-quick-construction-pack: 66 cards
[12/299] pokemon-2020-battle-academy
  pokemon-2020-battle-academy: 151 cards
[13/299] pokemon-ancient-origins
  pokemon-ancient-origins: 201 cards
[14/

,set,product_id,card,url,Ungraded,Grade 9,PSA 10
0,pokemon-1999-topps-movie,6777795,Ready For Battle,https://www.pricecharting.com/game/pokemon-199...,1416.85,NaN,NaN
1,pokemon-1999-topps-movie,3560907,Playtime [Corrected Foil] #57,https://www.pricecharting.com/game/pokemon-199...,NaN,NaN,NaN
2,pokemon-1999-topps-movie,3560860,Togepi in Trouble [Corrected] #45,https://www.pricecharting.com/game/pokemon-199...,NaN,NaN,NaN
3,pokemon-1999-topps-movie,3560886,Everyone Pull [Corrected Foil] #52,https://www.pricecharting.com/game/pokemon-199...,NaN,NaN,NaN
4,pokemon-1999-topps-movie,3560856,Looks Like Trouble [Corrected] #44,https://www.pricecharting.com/game/pokemon-199...,NaN,NaN,NaN


### 4. Price history per card \u2192 `pokemon_price_history.csv`

Each product page embeds `VGPC.chart_data`: monthly `[timestamp_ms, price_in_cents]`
points for six internal series. The series keys don't match their labels, so this
is the verified mapping (a `0` means "no sale that month" and is skipped).

In [5]:
# product-page series key -> the grade it actually represents
GRADE_TO_SERIES = {
    "Ungraded":  "used",
    "Grade 7":   "cib",
    "Grade 8":   "new",
    "Grade 9":   "graded",
    "Grade 9.5": "boxonly",
    "PSA 10":    "manualonly",
}

def scrape_history(product_url):
    html = fetch(product_url)
    m = re.search(r"VGPC\.chart_data\s*=\s*(\{.*?\})\s*;", html, re.S)
    if not m:
        return []
    chart = json.loads(m.group(1))
    out = []
    for grade, key in GRADE_TO_SERIES.items():
        for ts, cents in chart.get(key, []):
            if not cents:            # 0 = no data that month
                continue
            date = datetime.fromtimestamp(ts / 1000, tz=timezone.utc).strftime("%Y-%m-%d")
            out.append({"grade": grade, "date": date, "price": cents / 100.0})
    return out

# test on a single card
scrape_history("https://www.pricecharting.com/game/pokemon-promo/pikachu-with-grey-felt-hat-85")[:5]


[{'grade': 'Ungraded', 'date': '2023-09-01', 'price': 119.0},
 {'grade': 'Ungraded', 'date': '2023-10-01', 'price': 122.46},
 {'grade': 'Ungraded', 'date': '2023-11-01', 'price': 152.0},
 {'grade': 'Ungraded', 'date': '2023-12-01', 'price': 139.08},
 {'grade': 'Ungraded', 'date': '2024-01-01', 'price': 129.02}]

**Heads up:** history is one request per card, so the full dataset is tens of
thousands of requests. Test on a small slice first.

In [6]:
subset = prices.head(20)          # <-- start small; raise once you're happy

history = []
for i, row in enumerate(subset.itertuples(index=False), 1):
    print(f"[{i}/{len(subset)}] {row.card}")
    try:
        recs = scrape_history(row.url)
        for r in recs:
            r.update(card=row.card, set=row.set, product_id=row.product_id)
        history += recs
    except Exception as e:
        print("  !!", e)

hist_df = pd.DataFrame(history)
import os; os.makedirs("data", exist_ok=True)
hist_df.to_csv("data/pokemon_price_history.csv", index=False)
print("saved data/pokemon_price_history.csv", hist_df.shape)
hist_df.head()


[1/20] Ready For Battle
[2/20] Playtime [Corrected Foil] #57
[3/20] Togepi in Trouble [Corrected] #45
[4/20] Everyone Pull [Corrected Foil] #52
[5/20] Looks Like Trouble [Corrected] #44
[6/20] Better Than Ever [Corrected] #56
[7/20] Everyone Pull [Corrected] #52
[8/20] C'Mon We Need Help [Rainbow Foil] #59
[9/20] First Movie Booster Pack
[10/20] Pikachu's Vacation #42
[11/20] Booster Box
[12/20] Legends #1
[13/20] Trapped Charizard #51
[14/20] Farewell #39
[15/20] The Tail  End of the Race #50
[16/20] Fight Rages #35
[17/20] A Great Day [Corrected] #58
[18/20] Mechanical Mewtwo [Rainbow Foil] #6
[19/20] Future Looks Bright #41
[20/20] Trapped Charizard [Rainbow Foil] #51
saved pokemon_price_history.csv (2176, 6)


,grade,date,price,card,set,product_id
0,Ungraded,2025-08-01,1882.0,Ready For Battle,pokemon-1999-topps-movie,6777795
1,Ungraded,2025-09-01,1882.0,Ready For Battle,pokemon-1999-topps-movie,6777795
2,Ungraded,2025-10-01,1882.0,Ready For Battle,pokemon-1999-topps-movie,6777795
3,Ungraded,2025-12-01,1882.0,Ready For Battle,pokemon-1999-topps-movie,6777795
4,Ungraded,2026-01-01,1882.0,Ready For Battle,pokemon-1999-topps-movie,6777795
